# Combined correction system: timing → split → delete

This notebook chains the group's correction steps and scores the result. It **trains nothing**: every step's
model was already trained and exported by its own notebook, and this notebook only applies those exports in order.

| Order | Step | Owner | Export used (train = out-of-fold by song, val = final model) |
|---|---|---|---|
| 1 | Timing: move note starts/ends | Le Xin | `le_xin_timing_rf_audio_pitch_v2_nestedgate`, `le_xin_timing_val_rf_audio_pitch` |
| 2 | Split: cut notes that cover 2+ sung notes | Le Xin (replaces merge) | `le_xin_split_train_oof_v1`, `le_xin_split_val_v1` |
| 3 | Delete: remove extra (false) notes | Felicia | `felicia_false_corrected_train_oof_v2_exploratory`, `felicia_false_corrected_val_v2_exploratory` |

**Why this order.** Tested on the training songs (out-of-fold): timing → split → delete gave onset F1 0.3667,
delete → timing → split gave 0.3657 (timing retrained without the deleted notes). Nearly equal, because Felicia's
model decides on the original Basic Pitch notes either way; delete-last was kept because it scores higher and
needs no retraining. Hann Qian's pitch step is left out: on validation it added nothing (0.3796 → 0.3795).

**How the delete step is applied.** Felicia's export lists the notes she keeps. A note (and any pieces the split
step cut it into, `note_id~1`, `~2`, ...) is dropped when its original note ID is not in her kept list.

**Rules kept:** test data is never opened; answer keys are used for scoring only; every output is a COMPLETE note
table and source tables are never overwritten. Labels are provisional (proposed v2).

In [1]:
# Setup: paths, scorer, and the answer keys (scoring only, never model input)
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

HERE = next(p / 'notebooks' / 'correction_handover' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'selected_100.csv').exists())
ROOT = (HERE / '..' / '..').resolve()
sys.path[:0] = [str(HERE), str((HERE / '..').resolve())]
import correction_utils as cu
import bp_experiment as bx                                   # the project's reference parser (val answer key)
DATA, EXPORTS = HERE / '..' / '..' / 'basic_pitch_runs' / 'correction_handover', HERE / 'exports'

try:
    import librosa, mir_eval.transcription
    print('Scorer: real librosa + mir_eval')
except ImportError:
    sys.path.insert(0, str(ROOT / 'le_xin_work'))
    import shim
    print('Scorer: shim.py stand-in')

read = lambda path: pd.read_csv(path, dtype={'note_id':str, 'recording_id':str, 'sample_id':str})
all_manifest = pd.read_csv(DATA / 'recording_manifest.csv', dtype={'recording_id':str, 'sample_id':str})
MANIFEST = {'train': pd.read_csv(DATA / 'training_recording_manifest.csv', dtype={'recording_id':str, 'sample_id':str}),
            'val':   all_manifest[all_manifest['split'] == 'val'].reset_index(drop=True)}
for split, man in MANIFEST.items():
    assert man['split'].eq(split).all()                      # train and val only: test stays locked

val_events, _ = bx.references(MANIFEST['val'].to_dict('records'))
REFS = {'train': read(DATA / 'references_train.csv'),
        'val':   pd.DataFrame([dict(note_id=f'{rid}:ref{i:05d}', recording_id=rid, start=e[0], end=e[1], pitch=e[2])
                               for rid, events in val_events.items() for i, e in enumerate(events)])}
for split in MANIFEST:
    print(f"{split}: {MANIFEST[split].sample_id.nunique()} songs, {len(MANIFEST[split])} recordings, {len(REFS[split])} reference notes")

Scorer: real librosa + mir_eval
train: 60 songs, 534 recordings, 8040 reference notes
val: 20 songs, 160 recordings, 2501 reference notes


In [2]:
# Step 1: load each step's exported note table and check they fit together
FILES = {'train': dict(bp=DATA / 'train_features.csv',
                       timing=EXPORTS / 'le_xin_timing_rf_audio_pitch_v2_nestedgate.csv',
                       split=EXPORTS / 'le_xin_split_train_oof_v1.csv',
                       delete=EXPORTS / 'felicia_false_corrected_train_oof_v2_exploratory.csv'),
         'val':   dict(bp=DATA / 'val_features.csv',
                       timing=EXPORTS / 'le_xin_timing_val_rf_audio_pitch.csv',
                       split=EXPORTS / 'le_xin_split_val_v1.csv',
                       delete=EXPORTS / 'felicia_false_corrected_val_v2_exploratory.csv')}
original_id = lambda ids: ids.str.split('~').str[0]          # 'abc:n00006~1' -> 'abc:n00006'

tables, checks = {}, []
for split, files in FILES.items():
    t = {name: read(path) for name, path in files.items()}
    t['bp'] = t['bp'][cu.NOTE_COLUMNS]
    bp = t['bp'].set_index('note_id')
    for name, table in t.items():
        cu.validate_notes(table, MANIFEST[split])            # complete, well-formed note table for this split
    # timing: same notes as Basic Pitch, only start/end moved
    assert t['timing'].note_id.tolist() == t['bp'].note_id.tolist()
    assert np.array_equal(t['timing'].pitch, t['bp'].pitch)
    # split: every piece comes from a timing note, and every timing note is still there
    assert set(original_id(t['split'].note_id)) == set(t['timing'].note_id)
    # delete: Felicia only removes Basic Pitch notes; kept notes are unchanged
    kept = t['delete'].set_index('note_id')
    assert kept.index.isin(bp.index).all()
    assert np.allclose(kept[['start', 'end', 'pitch']], bp.loc[kept.index, ['start', 'end', 'pitch']])
    tables[split] = t
    checks.append(dict(split=split, bp_notes=len(t['bp']), after_timing=len(t['timing']),
                       after_split=len(t['split']), pieces_added=len(t['split']) - len(t['timing']),
                       felicia_keeps=len(kept), felicia_deletes=len(t['bp']) - len(kept)))
display(pd.DataFrame(checks).set_index('split'))
print('All inputs fit together: timing keeps every note, split pieces trace back to timing notes, '
      'Felicia only deletes.')

,bp_notes,after_timing,after_split,pieces_added,felicia_keeps,felicia_deletes
split,,,,,,
train,7623,7623,7982,359,7192,431
val,2329,2329,2437,108,2207,122


All inputs fit together: timing keeps every note, split pieces trace back to timing notes, Felicia only deletes.


In [3]:
# Step 2: run the chain on the training songs (all out-of-fold by song) and score after every stage
def apply_delete(table, kept_ids):
    """Drop every note, and every split piece of a note, whose original Basic Pitch ID Felicia did not keep."""
    return table[original_id(table['note_id']).isin(kept_ids)].reset_index(drop=True)

def chain(split):
    """The stages of the combined system for one split, in order. Each value is a COMPLETE note table."""
    t = tables[split]
    return {'Basic Pitch unchanged':              t['bp'],
            '+ timing':                           t['timing'],
            '+ timing + split':                   t['split'],
            '+ timing + split + delete (final)':  apply_delete(t['split'], set(t['delete']['note_id']))}

def score_stages(split, stages, extra=None):
    """Onset/offset F1 (song-macro, every recording) after each stage, and songs better/worse than the stage before."""
    rows, before = [], None
    for name, table in {**stages, **(extra or {})}.items():
        out = cu.evaluate_end_to_end(table[cu.NOTE_COLUMNS], REFS[split], MANIFEST[split])
        assert out['summary']['recordings'] == len(MANIFEST[split])
        song = {s['sample_id']: s['f1'] for s in out['songs']}
        row = dict(stage=name, notes=len(table),
                   onset_F1=round(out['summary']['song_macro_f1'], 4),
                   offset_F1=round(out['summary']['song_macro_offset_f1'], 4))
        if before is not None and name in stages:
            row['songs_better'] = sum(song[s] > before[s] + 1e-12 for s in song)
            row['songs_worse']  = sum(song[s] < before[s] - 1e-12 for s in song)
        if name in stages:
            before = song
        rows.append(row)
    table = pd.DataFrame(rows).set_index('stage')
    table['gain_vs_previous'] = table['onset_F1'].diff().round(4)
    return table

train_stages = chain('train')
train_results = score_stages('train', train_stages,
                             extra={'(reference) Felicia delete only': tables['train']['delete']})
train_results.loc['(reference) Felicia delete only', 'gain_vs_previous'] = np.nan
display(train_results)
print('songs_better / songs_worse: onset F1 per song vs the stage above (60 training songs).')

,notes,onset_F1,offset_F1,songs_better,songs_worse,gain_vs_previous
stage,,,,,,
Basic Pitch unchanged,7623,0.2628,0.1539,NaN,NaN,NaN
+ timing,7623,0.3541,0.2176,57.0,2.0,0.0913
+ timing + split,7982,0.3585,0.2243,36.0,20.0,0.0044
+ timing + split + delete (final),7550,0.3667,0.2294,38.0,2.0,0.0082
(reference) Felicia delete only,7192,0.2692,0.1579,NaN,NaN,NaN


songs_better / songs_worse: onset F1 per song vs the stage above (60 training songs).


In [4]:
# Step 3: the same chain, once, on the 20 validation songs (no model ever trained on them)
val_stages = chain('val')
val_results = score_stages('val', val_stages,
                           extra={'(reference) Felicia delete only': tables['val']['delete']})
val_results.loc['(reference) Felicia delete only', 'gain_vs_previous'] = np.nan
display(val_results)
print('songs_better / songs_worse: onset F1 per song vs the stage above (20 validation songs).')

# Train vs val side by side: does each stage help about as much on new songs?
side = pd.DataFrame({'train_onset_F1': train_results['onset_F1'], 'val_onset_F1': val_results['onset_F1'],
                     'train_gain': train_results['gain_vs_previous'], 'val_gain': val_results['gain_vs_previous'],
                     'train_offset_F1': train_results['offset_F1'], 'val_offset_F1': val_results['offset_F1']})
display(side)
total = {s: r.loc['+ timing + split + delete (final)', 'onset_F1'] - r.loc['Basic Pitch unchanged', 'onset_F1']
         for s, r in [('train', train_results), ('val', val_results)]}
print(f"Whole system, onset F1 gain over unchanged Basic Pitch: train +{total['train']:.4f}, val +{total['val']:.4f}")
print('Caveat: val was already used to choose the Basic Pitch settings, so it is continued development, not untouched evidence.')

,notes,onset_F1,offset_F1,songs_better,songs_worse,gain_vs_previous
stage,,,,,,
Basic Pitch unchanged,2329,0.2726,0.1602,NaN,NaN,NaN
+ timing,2329,0.3755,0.2360,20.0,0.0,0.1029
+ timing + split,2437,0.3796,0.2427,14.0,5.0,0.0041
+ timing + split + delete (final),2314,0.3904,0.2504,16.0,0.0,0.0108
(reference) Felicia delete only,2207,0.2806,0.1659,NaN,NaN,NaN


songs_better / songs_worse: onset F1 per song vs the stage above (20 validation songs).


,train_onset_F1,val_onset_F1,train_gain,val_gain,train_offset_F1,val_offset_F1
stage,,,,,,
Basic Pitch unchanged,0.2628,0.2726,NaN,NaN,0.1539,0.1602
+ timing,0.3541,0.3755,0.0913,0.1029,0.2176,0.2360
+ timing + split,0.3585,0.3796,0.0044,0.0041,0.2243,0.2427
+ timing + split + delete (final),0.3667,0.3904,0.0082,0.0108,0.2294,0.2504
(reference) Felicia delete only,0.2692,0.2806,NaN,NaN,0.1579,0.1659


Whole system, onset F1 gain over unchanged Basic Pitch: train +0.1039, val +0.1178
Caveat: val was already used to choose the Basic Pitch settings, so it is continued development, not untouched evidence.


In [5]:
# Step 4: export the final note tables + one action history per split (write once; a rerun checks they still match)
OUT = EXPORTS
STEP_INFO = {'timing': ('1 timing', 'Le Xin', 'timing_change'), 'split': ('2 split', 'Le Xin', 'split'),
             'delete': ('3 delete', 'Felicia', 'delete')}

def history(split):
    """Every change the chain made, in order: one row per moved note, per split note, per deleted note."""
    t, rows = tables[split], []
    bp, timed = t['bp'].set_index('note_id'), t['timing'].set_index('note_id')
    moved = ((timed.start - bp.start).abs() > 1e-9) | ((timed.end - bp.end).abs() > 1e-9)
    for nid in timed.index[moved]:
        rows.append(dict(note_id=nid, recording_id=timed.at[nid, 'recording_id'], stage='timing',
                         old_start=bp.at[nid, 'start'], new_start=timed.at[nid, 'start'],
                         old_end=bp.at[nid, 'end'], new_end=timed.at[nid, 'end']))
    pieces = t['split'].assign(orig=original_id(t['split'].note_id))
    for nid, g in pieces.groupby('orig'):
        if len(g) > 1:
            g = g.sort_values('start')
            rows.append(dict(note_id=nid, recording_id=g.recording_id.iloc[0], stage='split',
                             cut_times=';'.join(f'{x:.4f}' for x in g.start.iloc[1:]),
                             new_note_ids=';'.join(g.note_id.iloc[1:]),
                             new_pitches=';'.join(f'{p:g}' for p in g.pitch)))
    kept = set(t['delete'].note_id)
    for nid in bp.index[~bp.index.isin(kept)]:
        gone = pieces.note_id[pieces.orig == nid]
        rows.append(dict(note_id=nid, recording_id=bp.at[nid, 'recording_id'], stage='delete',
                         new_note_ids=';'.join(gone)))                       # the note and its pieces removed
    h = pd.DataFrame(rows)
    h.insert(2, 'step', h.stage.map(lambda s: STEP_INFO[s][0]))
    h.insert(3, 'action', h.stage.map(lambda s: STEP_INFO[s][2]))
    h.insert(4, 'owner', h.stage.map(lambda s: STEP_INFO[s][1]))
    h['label_version'] = 'proposed_v2/train_targets_v2.csv (provisional)'
    order = {nid: i for i, nid in enumerate(t['bp'].note_id)}
    return (h.assign(_o=h.note_id.map(order)).sort_values(['_o', 'step']).drop(columns=['_o', 'stage'])
             .reset_index(drop=True))

def save_once(table, path, man=None):
    if path.exists():
        saved = pd.read_csv(path, dtype={'note_id':str, 'recording_id':str})
        pd.testing.assert_frame_equal(saved, table.reset_index(drop=True), check_dtype=False)
        print('Already saved, unchanged:', path.name)
    elif man is not None:
        cu.export_corrections(table, man, path)              # validates the complete note table, never overwrites
        print('Saved:', path.name)
    else:
        with path.open('x', encoding='utf-8', newline='') as handle:
            table.to_csv(handle, index=False)
        print('Saved:', path.name)

summary = []
for split, stages, name in [('train', train_stages, 'combined_final_train_oof_v1'),
                            ('val',   val_stages,   'combined_final_val_v1')]:
    final = stages['+ timing + split + delete (final)'][cu.NOTE_COLUMNS]
    hist = history(split)
    # the history must explain the final table exactly
    pieces_added = hist.loc[hist.action == 'split', 'new_note_ids'].str.count(';').add(1).sum()
    rows_deleted = hist.loc[hist.action == 'delete', 'new_note_ids'].str.count(';').add(1).sum()
    assert len(final) == len(tables[split]['bp']) + pieces_added - rows_deleted
    assert set(original_id(final.note_id)) == set(tables[split]['bp'].note_id) - set(hist.loc[hist.action == 'delete', 'note_id'])
    save_once(final, OUT / f'{name}.csv', MANIFEST[split])
    save_once(hist, OUT / f'{name}_actions.csv')
    summary.append(dict(data=split, final_notes=len(final), **hist.action.value_counts().to_dict()))
display(pd.DataFrame(summary).set_index('data'))
display(history('val').head(8))

Saved: combined_final_train_oof_v1.csv
Saved: combined_final_train_oof_v1_actions.csv
Saved: combined_final_val_v1.csv
Saved: combined_final_val_v1_actions.csv


,final_notes,timing_change,delete,split
data,,,,
train,7550,5738,431,312
val,2314,1675,122,91


,note_id,recording_id,step,action,owner,old_start,new_start,old_end,new_end,cut_times,new_note_ids,new_pitches,label_version
0,990c3eef6a2382476daa:n00000,990c3eef6a2382476daa,1 timing,timing_change,Le Xin,0.069660,0.001593,0.243810,0.243810,NaN,NaN,NaN,proposed_v2/train_targets_v2.csv (provisional)
1,990c3eef6a2382476daa:n00001,990c3eef6a2382476daa,2 split,split,Le Xin,NaN,NaN,NaN,NaN,0.4876;0.5689,990c3eef6a2382476daa:n00001~1;990c3eef6a238247...,63;63;63,proposed_v2/train_targets_v2.csv (provisional)
2,990c3eef6a2382476daa:n00002,990c3eef6a2382476daa,1 timing,timing_change,Le Xin,1.137778,1.046177,1.369977,1.369977,NaN,NaN,NaN,proposed_v2/train_targets_v2.csv (provisional)
3,990c3eef6a2382476daa:n00003,990c3eef6a2382476daa,1 timing,timing_change,Le Xin,1.462857,1.390425,2.416159,2.416159,NaN,NaN,NaN,proposed_v2/train_targets_v2.csv (provisional)
4,990c3eef6a2382476daa:n00003,990c3eef6a2382476daa,2 split,split,Le Xin,NaN,NaN,NaN,NaN,1.6951;1.7531,990c3eef6a2382476daa:n00003~1;990c3eef6a238247...,63;63;63,proposed_v2/train_targets_v2.csv (provisional)
5,990c3eef6a2382476daa:n00005,990c3eef6a2382476daa,1 timing,timing_change,Le Xin,4.031230,3.987911,4.216990,4.377032,NaN,NaN,NaN,proposed_v2/train_targets_v2.csv (provisional)
6,990c3eef6a2382476daa:n00005,990c3eef6a2382476daa,3 delete,delete,Felicia,NaN,NaN,NaN,NaN,NaN,990c3eef6a2382476daa:n00005,NaN,proposed_v2/train_targets_v2.csv (provisional)
7,990c3eef6a2382476daa:n00006,990c3eef6a2382476daa,1 timing,timing_change,Le Xin,4.843928,4.744675,5.853996,5.853996,NaN,NaN,NaN,proposed_v2/train_targets_v2.csv (provisional)
